# Kedro + JupyterLab + MLflow: od danych do predykcji

Notebook jest klientem projektu. Korzysta z katalogu, parametrów i kodu z `src/`; nie zawiera drugiej implementacji treningu. Wybierz **Run → Run All Cells**. Utworzy źródło, jeśli jeszcze nie istnieje, wykona eksperymenty i zapisze nową partię predykcji.

Przykład: syntetyczne ceny nieruchomości w PLN. 1500 rekordów z targetem i 250 do predykcji. Dane są generowane lokalnie; nie pobieramy datasetu z internetu.

In [ ]:
from kedro_lasso.commands import prepare_runtime, seed
from kedro_lasso.settings import PROJECT_ROOT

prepare_runtime()
seed()
print(PROJECT_ROOT)

## 1. Wczytaj projekt Kedro

`demo lab` uruchamia JupyterLab przez `kedro jupyter lab`. W kernelu dostajesz `catalog`, `context`, `pipelines` i `session`. Jawne przeładowanie poniżej pozwala wykonać notebook także w zwykłym kernelu tego środowiska.

Po zmianie kodu lub YAML użyj `%reload_kedro`. Jedna sesja Kedro obsługuje jedno wykonanie; do kolejnych eksperymentów twórz nowe sesje.

In [ ]:
get_ipython().run_line_magic('load_ext', 'kedro.ipython')
get_ipython().run_line_magic('reload_kedro', f'"{PROJECT_ROOT}"')
print('Pipeline’y:', list(pipelines))
print('Datasety:', list(catalog.keys()))

## 2. Eksploruj dane przez katalog

Połączenie z SQLite i SQL są w `conf/base/catalog.yml` i `credentials.yml`. Notebook nie otwiera własnego, równoległego połączenia do źródła.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

raw = catalog.load('raw_training')
display(raw.head())
display(raw.describe().T)
display(raw.isna().sum().rename('braki').to_frame())

In [ ]:
raw.plot.scatter(x='area_m2', y='price_pln', alpha=0.25, figsize=(8, 4))
plt.title('Syntetyczne ceny nieruchomości')
plt.ylabel('Cena [PLN]')
plt.show()

## 3. Użyj tej samej funkcji co node Kedro

Eksploracja korzysta z kodu domenowego. Walidacja pilnuje kolumn, unikalnych ID, targetu i wartości nieskończonych. Braki cech są dozwolone: uzupełni je imputer dopasowany wyłącznie do danych treningowych.

In [ ]:
from kedro_lasso.domain import validate_frame

schema = context.params['data']
clean = validate_frame(raw, schema, training=True)
print('Kształt:', clean.shape)
print('Cechy:', schema['features'])
print('Domyślna siatka alpha:', context.params['model']['alphas'])

## 4. Uruchom cały flow z notebooka

Nadpisujemy tylko siatkę `alpha` i nazwę uruchomienia. Reszta konfiguracji pozostaje w YAML. To ten sam flow co `kedro run`: walidacja → podział 60/20/20 → eksperymenty → refit na train+validation → jednorazowa ocena testowa → scoring → transakcja w bazie wynikowej.

Zmiana siatki tworzy nowe uruchomienia MLflow oraz nową wersję modelu. Zbiór testowy nie uczestniczy w wyborze `alpha`.

In [ ]:
from kedro.framework.session import KedroSession

with KedroSession.create(
    project_path=PROJECT_ROOT,
    runtime_params={
        'model': {'alphas': [0.1, 10.0, 100.0, 1000.0, 10000.0]},
        'tracking': {'run_name': 'notebook-sweep'},
    },
) as experiment_session:
    experiment_session.run()

# Nowy kontekst widzi najnowszą zapisaną wersję modelu i raportu.
get_ipython().run_line_magic('reload_kedro', f'"{PROJECT_ROOT}"')
bundle = catalog.load('model_bundle')
report = bundle['report']
display(pd.DataFrame(report['trials']).sort_values('rmse'))
display(pd.DataFrame({
    'Lasso test': report['test_metrics'],
    'Baseline test': report['baseline_test_metrics'],
}))
print('Najlepsze alpha:', report['best_alpha'])
print('MLflow run_id:', report['run_id'])
print('Model URI:', report['model_uri'])

## 5. Porównaj eksperymenty MLflow

W UI pod adresem http://127.0.0.1:5000 otwórz eksperyment `housing-lasso`. Uruchomienie nadrzędne zawiera metryki testowe, konfigurację, identyfikatory podziału i model z sygnaturą wejścia. Uruchomienia podrzędne odpowiadają wartościom `alpha` i zawierają metryki walidacyjne.

In [ ]:
import mlflow
from kedro_lasso.config import load_config

tracking = load_config()['parameters']['tracking']
mlflow.set_tracking_uri(tracking['uri'])
runs = mlflow.search_runs(experiment_names=[tracking['experiment_name']])
columns = [column for column in [
    'run_id', 'tags.mlflow.runName', 'tags.mlflow.parentRunId',
    'params.alpha', 'params.best_alpha', 'metrics.validation_rmse',
    'metrics.test_rmse', 'status',
] if column in runs.columns]
display(runs[columns].head(20))

In [ ]:
trials = pd.DataFrame(report['trials']).sort_values('alpha')
ax = trials.plot(x='alpha', y='rmse', marker='o', logx=True, legend=False, figsize=(8, 4))
ax.set(xlabel='alpha (skala log)', ylabel='Walidacyjny RMSE [PLN]',
       title='Wybór regularyzacji na walidacji')
plt.show()

coefficients = pd.Series(bundle['model']['lasso'].coef_, index=report['schema']['features'])
coefficients.sort_values().plot.barh(figsize=(8, 4))
plt.xlabel('Współczynnik po standaryzacji [PLN]')
plt.title('Które cechy zachowało Lasso?')
plt.show()

## 6. Wyniki z drugiej bazy danych

`warehouse.db` ma cztery tabele: `model_runs`, `experiment_trials`, `prediction_batches`, `predictions`. Partia ma własne ID, fingerprint wejścia i powiązanie z runem MLflow. Poniżej czytamy wyniki przez katalog Kedro.

In [ ]:
predictions = catalog.load('warehouse_predictions')
display(predictions.head(10))
display(predictions.groupby('batch_id').agg(n_rows=('property_id', 'size')))
assert len(predictions) >= 250
assert predictions['prediction'].notna().all()

## 7. Scoring bez ponownego treningu

`pipeline_name='scoring'` wczytuje najnowszą wersję modelu. Używa zapisanej listy cech i dopasowanego preprocessingu. Nie tworzy nowych eksperymentów treningowych. Powstaje nowa partia predykcji, a wcześniejsze wyniki pozostają w bazie.

In [ ]:
runs_before = len(mlflow.search_runs(experiment_names=[tracking['experiment_name']]))
with KedroSession.create(project_path=PROJECT_ROOT) as scoring_session:
    scoring_session.run(pipeline_name='scoring')
runs_after = len(mlflow.search_runs(experiment_names=[tracking['experiment_name']]))
assert runs_before == runs_after
get_ipython().run_line_magic('reload_kedro', f'"{PROJECT_ROOT}"')
display(catalog.load('warehouse_predictions').groupby('batch_id').size().rename('n_rows'))

## 8. Odtwórz model bez Kedro

Artefakt MLflow zawiera imputer, scaler i model. Służy do niezależnej predykcji w innym procesie. Ładuj wyłącznie zaufane artefakty: lokalny katalog Kedro i ten model MLflow korzystają z serializacji Python.

In [ ]:
import mlflow.sklearn
import numpy as np

restored_model = mlflow.sklearn.load_model(report['model_uri'])
scoring_data = catalog.load('raw_scoring')
features = report['schema']['features']
restored_predictions = restored_model.predict(scoring_data[features])
np.testing.assert_allclose(restored_predictions, bundle['model'].predict(scoring_data[features]))
display(pd.DataFrame({
    'property_id': scoring_data['property_id'],
    'prediction_pln': restored_predictions,
}).head())

## Co zmieniać dalej

- Cechy i hiperparametry: `conf/base/parameters.yml` albo `runtime_params` nowej sesji.
- Szybki trening: `kedro run --env quick --pipeline training`.
- Inne źródło danych: połączenie `source_db` i SQL w katalogu; zachowaj kontrakt kolumn.
- Inna baza wynikowa: połączenie `warehouse_db` i odpowiedni sterownik SQLAlchemy. Adapter ma jawny schemat, klucze i transakcję.
- Opcjonalny graf: `demo viz`, potem http://127.0.0.1:4141.

Opis granic projektu, schematu baz i sposobów rozszerzania: `docs/architecture.md`. Po wielokrotnym eksperymentowaniu na stałym zbiorze testowym końcowy model należy sprawdzić na nowym, niezależnym zbiorze.